In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from copy import deepcopy
import itertools
import numpy as np
from tqdm import tqdm

from odd_remez import remez
from visualization import visualization


def get_num_flops(n, m, deg):
  if n > m:
    n, m = m, n
  if deg == 0:
    return 0
  return 3 * n ** 2 * m + (deg // 2 - 1) * n ** 3

In [47]:
# most aggressive range of seqeunces for largest n/m ratio
def prepare_sequences(deg_arr, n, m):
    deg_2_flops = {deg: get_num_flops(n, m, deg) for deg in deg_arr}
    max_seq_len = 0
    while deg_2_flops[3] * (max_seq_len + 1) < deg_2_flops[5] * 5:
        max_seq_len += 1

    polynomial_seq_arr = set()
    for seq_len in range(1, max_seq_len + 1):
        for seq in itertools.product(deg_arr, repeat=seq_len):
            curr_flops = 0
            for deg in seq:
                curr_flops += deg_2_flops[deg]
            if curr_flops < deg_2_flops[5] * 5 or seq == (5, 5, 5, 5, 5):
                polynomial_seq_arr.add(seq)

    return polynomial_seq_arr

def grid_search_all_sequences(polynomial_seq_arr, left_start=0.001, lower_safety_factor=0.02407327424182761, upper_safety_factor=1.01):
    seq_2_errors = {}
    seq_2_polynomials_str = {}
    seq_2_polynomials = {}

    for seq in tqdm(polynomial_seq_arr):
        try:
            polys = []
            polys_str = []
            errors = []
            left = left_start
            right = 1.0
            for i in range(len(seq)):
                if seq[:i+1] in seq_2_errors:
                    errors = deepcopy(seq_2_errors[seq[:i+1]])
                    polys_str = deepcopy(seq_2_polynomials_str[seq[:i+1]])
                    polys = deepcopy(seq_2_polynomials[seq[:i+1]])
                    left = 1 - errors[-1]
                    right = 1 + errors[-1]
                else:
                    fx = "np.ones_like(x)"
                    fx_der = "np.zeros_like(x)"
                    n = seq[i] // 2
                    input_left = max(left, right * lower_safety_factor)
                    # input_left = left
                    
                    interval = [input_left, right]
                    px, xn, history = remez(fx, fx_der, interval, n, safety_factor=upper_safety_factor)
                    an = history['an'][-1]
                    left_degs = np.array([left ** (2 * i + 1) for i in range(len(an))])
                    left_error = 1 - np.dot(left_degs, an)
                    
                    curr_error = max(history['e'][-1], left_error)
                    errors.append(curr_error)
                    polys_str.append(px)
                    polys.append(an)
                    left = 1 - errors[-1]
                    right = 1 + errors[-1]
                    seq_2_errors[seq[:i+1]] = deepcopy(errors)
                    seq_2_polynomials_str[seq[:i+1]] = deepcopy(polys_str)
                    seq_2_polynomials[seq[:i+1]] = deepcopy(polys)
        except:
            continue
    return seq_2_errors, seq_2_polynomials, seq_2_polynomials_str

def get_seq_2_flops(n, m, polynomial_seq_arr, deg_arr):
    deg_2_flops = {deg: get_num_flops(n, m, deg) for deg in deg_arr}
    seq_2_flops = {}
    for seq in polynomial_seq_arr:
        curr_flops = 0
        for deg in seq:
            curr_flops += deg_2_flops[deg]
        seq_2_flops[seq] = curr_flops
    return seq_2_flops

def find_seqs_with_large_coefficients(seq_2_polynomials, threshold=10000.0):
    keys_to_delete = set()
    for key, value in seq_2_polynomials.items():
        for cur_coefs in value:
            if (np.abs(cur_coefs) > threshold).any():
                keys_to_delete.add(key)
                break
    return keys_to_delete

def find_seqs_with_large_error(seq_2_errors, threshold=0.35):
    keys_to_delete = set()
    for key, value in seq_2_errors.items():
        if value[-1].item() > threshold:
            keys_to_delete.add(key)
    return keys_to_delete

def find_seqs_with_large_deg(seqs, deg):
    keys_to_delete = set()
    for seq in seqs:
        if any(deg <= x for x in seq):
            keys_to_delete.add(seq)
    return keys_to_delete

def find_seqs_with_large_flops(seq_2_flops, threshold):
    keys_to_delete = set()
    for seq, value in seq_2_flops.items():
        if value > threshold:
            keys_to_delete.add(seq)
    return keys_to_delete

def find_optimal_keys(error_dict, cost_dict):
    keys = list(error_dict.keys())
    optimal_keys = []
    
    for key1 in keys:
        is_dominated = False
        for key2 in keys:
            if key1 != key2:
                # if (error_dict[key2] <= error_dict[key1] and 
                #     cost_dict[key2] <= cost_dict[key1] and
                #     (error_dict[key2] < error_dict[key1] or 
                #      cost_dict[key2] < cost_dict[key1])):

                if (error_dict[key2] < error_dict[key1] and 
                    cost_dict[key2] < cost_dict[key1]):
                    is_dominated = True
                    break
        
        if not is_dominated:
            optimal_keys.append(key1)
    
    return optimal_keys

def get_optimal_keys(seq_2_errors, seq_2_polynomials, seq_2_flops, error_threshold=0.35, coeff_threshold=10000.0):
    keys_with_large_coefs = find_seqs_with_large_coefficients(seq_2_polynomials, threshold=coeff_threshold)
    keys_with_large_error = find_seqs_with_large_error(seq_2_errors, threshold=error_threshold)
    keys_with_large_flops = find_seqs_with_large_flops(seq_2_flops, threshold=seq_2_flops[(5, 5, 5, 5, 5)])
    keys_to_delete = keys_with_large_coefs | keys_with_large_error | keys_with_large_flops

    seq_2_errors_cleaned = {k: v for k, v in seq_2_errors.items() if k not in keys_to_delete}
    seq_2_flops_cleaned = {k: v for k, v in seq_2_flops.items() if k not in keys_to_delete}
    seq_2_last_error = {k: v[-1].item() for k, v in seq_2_errors_cleaned.items()}
    optimal_keys = find_optimal_keys(seq_2_last_error, seq_2_flops_cleaned)
    return optimal_keys

In [11]:
def find_optimal_sequences_by_shape(shapes, left=0.001, max_deg=10, error_threshold=0.35, coeff_threshold=10_000.0, lower_safety_factor=0.02407327424182761, upper_safety_factor=1.01):
    deg_arr = [2 * i + 1 for i in range((max_deg + 1) // 2)]

    polynomial_seq_arr = set()
    for shape in shapes:
        n, m = shape
        polynomial_seq_arr |= prepare_sequences(deg_arr, n, m)

    seq_2_errors, seq_2_polynomials, seq_2_polynomials_str = grid_search_all_sequences(polynomial_seq_arr=polynomial_seq_arr, left_start=left, lower_safety_factor=lower_safety_factor, upper_safety_factor=upper_safety_factor)

    optimal_dicts = {}
    for shape in shapes:
        n, m = shape
        if n > m:
            n, m = m, n
        
        seq_2_flops = get_seq_2_flops(n, m, polynomial_seq_arr=polynomial_seq_arr, deg_arr=deg_arr)
        optimal_keys = get_optimal_keys(seq_2_errors, seq_2_polynomials, seq_2_flops, error_threshold=error_threshold, coeff_threshold=coeff_threshold)
        optimal_keys.sort(key=lambda x: seq_2_flops[x])
        
        optimal_dict = {
            key: {
                "flops": seq_2_flops[key],
                "flops_ratio": seq_2_flops[key] / seq_2_flops[(5, 5, 5, 5, 5)],
                "error": seq_2_errors[key][-1].item(),
                "polynomials": seq_2_polynomials[key],
                "polynomials_str": seq_2_polynomials_str[key],
            } for key in optimal_keys
        }
        optimal_dicts[shape] = optimal_dict
    return optimal_dicts

In [39]:
optimal_dict = find_optimal_sequences_by_shape([(2**12, 2**12 * 3), (2**12, 2**12), (2**12, 2**12 // 4)], max_deg=18, error_threshold=0.4, coeff_threshold=100_000.0)

100%|██████████| 9789/9789 [24:58<00:00,  6.53it/s]  


In [18]:
for v in optimal_dict.values():
    print(len(v))

28


In [ ]:
for optimal in optimal_dict.values():
    # print(type(optimal), len(optimal))
    optimal_keys = list(optimal.keys())
    optimal_keys.sort(key=lambda k: optimal[k]['flops_ratio'])
    for key, value in optimal.items():
        print(key, value['error'], max(np.max(np.abs(poly)) for poly in value['polynomials']), value['flops_ratio'])
    print()

(7, 5, 5, 9) 0.35263709488749584 97.4153883755767 0.86
(5, 7, 7, 7) 0.33074320668621127 20.90656923658714 0.86
(5, 5, 7, 9) 0.35484975399495766 20.90656923658714 0.86
(5, 7, 5, 9) 0.35310077771062554 20.90656923658714 0.86
(7, 7, 7, 5) 0.33237860077488823 97.4153883755767 0.86
(7, 5, 7, 7) 0.3302696287251802 97.4153883755767 0.86
(7, 7, 5, 7) 0.3306306628121296 97.4153883755767 0.86
(7, 3, 5, 13) 0.08712730795229628 47383.01144357805 0.88
(7, 7, 5, 9) 0.2506900780129444 97.4153883755767 0.88
(7, 7, 3, 11) 0.023016996499342524 83465.51496320935 0.88
(3, 7, 5, 13) 0.08759079822428806 47598.269598486324 0.88
(5, 3, 7, 13) 0.08788639694344602 47735.46926386558 0.88
(5, 7, 7, 9) 0.2507951300462149 20.90656923658714 0.88
(5, 5, 7, 11) 0.014424918467043524 51680.94437084821 0.88
(5, 7, 5, 11) 0.014200430574418443 50856.93734688892 0.88
(5, 5, 5, 13) 0.06168105826947734 35339.99833503735 0.88
(7, 3, 7, 11) 0.023602475388279753 85645.93217991863 0.88
(7, 7, 7, 7) 0.2276991642688586 97.415388375

In [45]:
for value in optimal_dict.values():
    for poly in value[(5, 7, 7, 7)]['polynomials']:
        print(poly)
    for poly in value[(5, 7, 7, 7)]['polynomials_str']:
        print(poly)
    break

[  7.49027339 -20.90656924  15.02655934]
[ 5.01105993 -6.90058076  3.10188215 -0.41882112]
[ 5.08776791 -7.22235326  3.34667675 -0.46581389]
[ 3.68956046 -4.98338722  2.52754141 -0.39914443]
15.026559337174017 * x**5 - 20.90656923658714 * x**3 + 7.490273393478118 * x
- 0.41882111943633665 * x**7 + 3.101882145128696 * x**5 - 6.900580758447274 * x**3 + 5.011059931358244 * x
- 0.46581389194702894 * x**7 + 3.3466767497580996 * x**5 - 7.222353263169536 * x**3 + 5.087767907791593 * x
- 0.39914443187258075 * x**7 + 2.527541407733747 * x**5 - 4.9833872244791495 * x**3 + 3.6895604572396943 * x


In [ ]:
def format_polynomial_for_desmos(polynomial_str, function_name):
    """
    Преобразует полином из Python формата в формат Desmos
    
    Args:
        polynomial_str: строка с полиномом в формате Python
        function_name: имя функции для Desmos (например, "f_{41}")
    """
    # Заменяем ** на ^
    result = polynomial_str.replace('**', '^')
    
    # Убираем пробелы вокруг операторов для более компактного вида
    result = result.replace(' * ', '')
    result = result.replace(' + ', '+')
    result = result.replace(' - ', '-')
    
    # Добавляем имя функции
    result = f"{function_name}(x)={result}"
    
    return result

# Исходные полиномы
polynomials = [
    "15.026559337174017 * x**5 - 20.90656923658714 * x**3 + 7.490273393478118 * x",
"-0.41882111943633665 * x**7 + 3.101882145128696 * x**5 - 6.900580758447274 * x**3 + 5.011059931358244 * x",
"-0.46581389194702894 * x**7 + 3.3466767497580996 * x**5 - 7.222353263169536 * x**3 + 5.087767907791593 * x",
"-0.39914443187258075 * x**7 + 2.527541407733747 * x**5 - 4.9833872244791495 * x**3 + 3.6895604572396943 * x"
]

# Форматируем каждый полином
for i, poly in enumerate(polynomials, 1):
    function_name = f"f_{{{4}{i}}}"
    formatted = format_polynomial_for_desmos(poly, function_name)
    print(formatted)


In [50]:
import re

def parse_polynomial_coefficients_odd_only(polynomial_str):
    """
    Извлекает коэффициенты полинома только для нечетных степеней
    
    Args:
        polynomial_str: строка с полиномом
        
    Returns:
        tuple: коэффициенты для степеней 1, 3, 5, 7, ... (только нечетные)
    """
    # Словарь для хранения коэффициентов по степеням
    coefficients = {}
    
    # Заменяем пробелы для более простой обработки
    clean_str = polynomial_str.replace(' ', '')
    
    # Добавляем + в начало если строка не начинается с -
    if not clean_str.startswith('-'):
        clean_str = '+' + clean_str
    
    # Находим все термы
    terms = re.findall(r'[+-][^+-]+', clean_str)
    
    for term in terms:
        term = term.strip()
        
        if 'x**' in term:
            # Терм вида коэффициент*x**степень
            parts = term.split('*x**')
            coeff = float(parts[0].replace('*', ''))
            power = int(parts[1])
            if power % 2 == 1:  # Только нечетные степени
                coefficients[power] = coeff
                
        elif term.endswith('*x') or (term.endswith('x') and not term.endswith('**x')):
            # Терм вида коэффициент*x (степень 1)
            if '*x' in term:
                coeff = float(term.replace('*x', ''))
            else:
                coeff = float(term.replace('x', ''))
            coefficients[1] = coeff  # Степень 1 - нечетная
    
    # Находим максимальную нечетную степень
    odd_powers = [p for p in coefficients.keys() if p % 2 == 1]
    max_odd_power = max(odd_powers) if odd_powers else 1
    
    # Создаем tuple коэффициентов для нечетных степеней от 1 до max_odd_power
    result = []
    for i in range(1, max_odd_power + 1, 2):  # 1, 3, 5, 7, ...
        result.append(coefficients.get(i, 0.0))
    
    return tuple(result)

# Исходные полиномы
polynomials = [
    "15.026559337174017 * x**5 - 20.90656923658714 * x**3 + 7.490273393478118 * x",
"-0.41882111943633665 * x**7 + 3.101882145128696 * x**5 - 6.900580758447274 * x**3 + 5.011059931358244 * x",
"-0.46581389194702894 * x**7 + 3.3466767497580996 * x**5 - 7.222353263169536 * x**3 + 5.087767907791593 * x",
"-0.39914443187258075 * x**7 + 2.527541407733747 * x**5 - 4.9833872244791495 * x**3 + 3.6895604572396943 * x"
]


# Парсим каждый полином
coefficient_tuples = []
for poly in polynomials:
    coeffs = parse_polynomial_coefficients_odd_only(poly)
    coefficient_tuples.append(coeffs)
    print(f"Полином: {poly}")
    print(f"Коэффициенты (только нечетные степени): {coeffs}")
    print()

print("Список tuples:")
print(coefficient_tuples)


Полином: 15.026559337174017 * x**5 - 20.90656923658714 * x**3 + 7.490273393478118 * x
Коэффициенты (только нечетные степени): (7.490273393478118, -20.90656923658714, 15.026559337174017)

Полином: -0.41882111943633665 * x**7 + 3.101882145128696 * x**5 - 6.900580758447274 * x**3 + 5.011059931358244 * x
Коэффициенты (только нечетные степени): (5.011059931358244, -6.900580758447274, 3.101882145128696, -0.41882111943633665)

Полином: -0.46581389194702894 * x**7 + 3.3466767497580996 * x**5 - 7.222353263169536 * x**3 + 5.087767907791593 * x
Коэффициенты (только нечетные степени): (5.087767907791593, -7.222353263169536, 3.3466767497580996, -0.46581389194702894)

Полином: -0.39914443187258075 * x**7 + 2.527541407733747 * x**5 - 4.9833872244791495 * x**3 + 3.6895604572396943 * x
Коэффициенты (только нечетные степени): (3.6895604572396943, -4.9833872244791495, 2.527541407733747, -0.39914443187258075)

Список tuples:
[(7.490273393478118, -20.90656923658714, 15.026559337174017), (5.011059931358244,

In [48]:
optimal_dict = find_optimal_sequences_by_shape([(2**12, 2**12 * 3), (2**12, 2**12), (2**12, 2**12 // 4)], left=0.0015, max_deg=18, error_threshold=0.4, coeff_threshold=100.0)

100%|██████████| 9789/9789 [23:27<00:00,  6.95it/s]  


In [49]:
for optimal in optimal_dict.values():
    # print(type(optimal), len(optimal))
    optimal_keys = list(optimal.keys())
    optimal_keys.sort(key=lambda k: optimal[k]['flops_ratio'])
    for key, value in optimal.items():
        print(key, value['error'], max(np.max(np.abs(poly)) for poly in value['polynomials']), value['flops_ratio'])
    print()

(5, 5, 7, 5) 0.3949420519420702 20.90656923658714 0.82
(5, 5, 5, 7) 0.39173859548437107 20.90656923658714 0.82
(3, 7, 7, 7) 0.34673283292148926 7.205171218286531 0.84
(7, 7, 3, 7) 0.353568647223478 97.4153883755767 0.84
(3, 7, 5, 9) 0.36874441731978047 10.463690280168462 0.84
(7, 5, 3, 9) 0.3687128944020207 97.4153883755767 0.84
(5, 7, 7, 5) 0.29330556683408204 20.90656923658714 0.84
(5, 5, 7, 7) 0.28577952593790823 20.90656923658714 0.84
(5, 7, 5, 7) 0.291405218583601 20.90656923658714 0.84
(5, 5, 5, 9) 0.30897365513648145 20.90656923658714 0.84
(7, 7, 7, 3) 0.3575760205623265 97.4153883755767 0.84
(7, 5, 7, 5) 0.29269031217847363 97.4153883755767 0.84
(5, 3, 7, 9) 0.3696706325032191 20.90656923658714 0.84
(3, 5, 7, 9) 0.37041838207933164 10.48636034069862 0.84
(7, 3, 7, 7) 0.34525256710409513 97.4153883755767 0.84
(5, 7, 3, 9) 0.36935944856551395 20.90656923658714 0.84
(7, 3, 5, 9) 0.3672971229012132 97.4153883755767 0.84
(7, 7, 5, 5) 0.3031250000078125 97.4153883755767 0.84
(7, 5, 5

In [ ]:
for value in optimal_dict.values():
    